# Caso 1: EH2025 y componente municipal del Censo 2024

No asistencia educativa de 12 a 17 años. Microdatos reales EH2025 recibidos el 15/11/2026. Los modelos individuales utilizan EH2025 y el agrupamiento municipal utiliza tabulados públicos del Censo 2024 y cartografía UDAPE. No se asignan municipios a personas EH ni se interpretan diferencias entre fuentes como cambios anuales. No mide abandono longitudinal ni efectos causales.


## Preparar la carpeta completa

Usar un kernel Python3.12 y abrir la carpeta que contiene src/, data/ y config.json. En otro equipo, instalar requirements.txt en un entorno propio. Se requiere el acceso autorizado a los módulos Persona y Vivienda; el paquete público no incluye registros individuales.

In [ ]:
from pathlib import Path
import json, subprocess, sys
base = Path.cwd().resolve()
candidates = [base, base.parent, base / "Proyecto_EH2025_ML_Caso1"]
ROOT = next((p for p in candidates if (p/"src").is_dir() and (p/"config.json").is_file()), None)
assert ROOT is not None, "Abre la carpeta completa del proyecto."
assert sys.version_info[:2] == (3, 12), "El entorno comprobado utiliza Python 3.12."
print("Proyecto:", ROOT.name)


## Instalar dependencias cuando sea necesario

Activar la siguiente opción solo en un entorno preparado para este trabajo. La instalación requiere conexión; los gráficos incluidos funcionan localmente.

In [ ]:
INSTALL_REQUIREMENTS = False
if INSTALL_REQUIREMENTS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(ROOT/"requirements.txt")], check=True)


## Comprobar los archivos originales

Guardar las copias según data/source_manifest.json. El módulo Vivienda conserva el sufijo_1. Registrar la procedencia y fecha reales si se descargan nuevamente. No imprimir filas ni identificadores.

In [ ]:
manifest = json.loads((ROOT/"data/source_manifest.json").read_text())
paths = manifest["raw_module_files"]
missing = [name for name, relative in paths.items() if not (ROOT/relative).is_file()]
assert not missing, "Faltan módulos en data/raw: " + ", ".join(missing)
def run_stage(stage):
    subprocess.run([sys.executable, str(ROOT/"run.py"), stage], cwd=ROOT, check=True)
run_stage("preflight")


## Ingesta y preparación

Lee SPSS conservando códigos, cruza con DuckDB y controla filas, ponderadores y claves. Mantiene el universo de 12 a 17 años y excluye predictores con fuga.

In [ ]:
run_stage("prepare")


## Reporte exploratorio automatizado

Este reporte describe la muestra sin ponderar. Las tasas y sus intervalos se calculan por separado con el diseño de encuesta.

In [ ]:
run_stage("profile")


## Modelos y evaluación

Random Forest y XGBoost con reserva y CV agrupadas por UPM. Imputación y one-hot se ajustan dentro de cada entrenamiento. El umbral 0,5 se fija antes de consultar la reserva. No modificarlo para mejorar artificialmente el resultado de prueba.

In [ ]:
run_stage("model")


## Descriptivos EH2025 y agrupamiento individual adicional

Incluye convención adjust para dos estratos solitarios; los IC estrictos no son estimables. La precisión geográfica es limitada. Se crean siete gráficos y perfiles individuales KMeans sin usar el objetivo para formar grupos. Este es un análisis adicional; el componente municipal se desarrolla a continuación con otra fuente.


In [ ]:
run_stage("visualize")


## Componente municipal con Censo 2024

Reextrae los seis tabulados oficiales incluidos y verifica las 343 equivalencias con la cartografía UDAPE. Ejecuta KMeans con cuatro indicadores estandarizados y peso igual por jurisdicción. La no asistencia participa en el agrupamiento: las diferencias entre perfiles son descriptivas y no validan un modelo predictivo. Se guarda el dataset real en outputs/municipal/ como CSV y Parquet, más mapa, PCA y perfiles interactivos.


In [ ]:
subprocess.run([sys.executable, str(ROOT/"data/preparar_municipios_censo2024.py"), "--data-dir", str(ROOT/"data/municipal"), "--geojson", str(ROOT/"data/bolivia_municipios_2024.geojson")], cwd=ROOT, check=True)
run_stage("municipal")


## Dataset individual procesado para evaluación con acceso controlado

Exporta datos reales minimizados, versión con diseño y plan muestral, diccionario y auditorías. Conserva nulos y tipos; no incluye folio ni número de persona. La carpeta outputs/entrega_restringida/ está excluida de GitHub y se entrega por separado conforme al protocolo de ANDA.


In [ ]:
run_stage("deliver-dataset")


## Sensibilidad y explorador de resultados

Excluye secundaria completa/superior como sensibilidad descriptiva, sin reentrenar. Actualiza el portal y la auditoría. El informe DOCX/PDF incluido fue generado desde estos agregados y revisado aparte. Para reconstruir el Word, ejecutar python -m src.build_report después de este cuaderno; exportar después a PDF desde Word o LibreOffice.


In [ ]:
subprocess.run([sys.executable, "-m", "src.finalize"], cwd=ROOT, check=True)
summary = json.loads((ROOT/"outputs/descriptive/survey_summary.json").read_text())
metrics = json.loads((ROOT/"outputs/model_evaluation/model_metrics.json").read_text())
print("Tasa ponderada (%)", summary["national"]["tasa_pct"])
print("Modelo seleccionado por CV:", metrics["selected_model"])
print("Abrir docs/Explorar_resultados.html y docs/Informe_Caso1_EH2025_Final.docx")


## Entrega

Leer docs/Entrega_en_GitHub.md. El componente municipal y los dos datasets procesados ya están desarrollados. Publicar únicamente el ZIP para GitHub, que contiene datos municipales agregados. Conservar el complemento individual como entrega académica separada de acceso controlado. La publicación y el envío a Moodle no se han realizado.

Fuente: Instituto Nacional de Estadística, Encuesta de Hogares 2025 - http://anda.ine.gob.bo/index.php/catalog/256. Fecha de acceso: 15 de noviembre de 2026. Complemento: INE Censo 2024 y UDAPE Atlas 2025, fuentes en data/municipal/.
